In [2]:
import torch
import torch.nn.functional as F
import math

def scaled_dot_product_attention(Q, K, V):
    # Узнаем размерность вектора (d_k), чтобы извлечь из неё корень
    d_k = Q.size(-1) 
    
    # 1. Скалярное произведение Запросов и Ключей: Q * K^T
    # transpose(-2, -1) переворачивает только две последние размерности матрицы (строки и столбцы)
    scores = torch.bmm(Q, K.transpose(-2, -1))
    
    # 2. Масштабирование
    scores = scores / math.sqrt(d_k)
    
    # 3. Softmax по последнему измерению (по строкам)
    weights = F.softmax(scores, dim=-1)
    
    # 4. Умножение на Значения
    output = torch.bmm(weights, V)
    
    return output, weights

# Проверка работы:
batch_size, seq_len, d_k = 2, 5, 64
q = torch.randn(batch_size, seq_len, d_k)
k = torch.randn(batch_size, seq_len, d_k)
v = torch.randn(batch_size, seq_len, d_k)

out, attn_weights = scaled_dot_product_attention(q, k, v)
print("Форма выхода:", out.shape)          # [2, 5, 64]
print("Форма весов внимания:", attn_weights.shape) # [2, 5, 5] (матрица связей 5 слов с 5 словами)

Форма выхода: torch.Size([2, 5, 64])
Форма весов внимания: torch.Size([2, 5, 5])


In [3]:
import torch.nn as nn

class CustomMultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        
        # Размерность одной головы (например, 512 / 8 = 64)
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim, "embed_dim должен делиться на num_heads нацело"
        
        # Линейные слои (формочки) для создания Q, K, V из входного текста
        # Заметь: мы создаем одну большую матрицу для всех голов сразу, так работает быстрее
        self.W_q = nn.Linear(embed_dim, embed_dim)
        self.W_k = nn.Linear(embed_dim, embed_dim)
        self.W_v = nn.Linear(embed_dim, embed_dim)
        
        # Финальный слой для обработки склеенного результата (W_o)
        self.W_o = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        
        # 1. Лепим Q, K, V из входных эмбеддингов
        Q = self.W_q(x)  # [batch_size, seq_len, embed_dim]
        K = self.W_k(x)
        V = self.W_v(x)
        
        # 2. Делим на Головы (Splitting into heads)
        # Меняем форму тензора: разбиваем embed_dim на (num_heads, head_dim)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim)
        
        # Транспонируем, чтобы num_heads и seq_len поменялись местами для удобного умножения
        # Новая форма: [batch_size, num_heads, seq_len, head_dim]
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        
        # 3. Вычисляем Attention (формула та же, но torch.matmul сам работает с 4D тензорами)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        weights = F.softmax(scores, dim=-1)
        context = torch.matmul(weights, V) # [batch_size, num_heads, seq_len, head_dim]
        
        # 4. Склеиваем головы обратно (Concatenation)
        # Собираем измерения обратно в [batch_size, seq_len, embed_dim]
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.embed_dim)
        
        # 5. Пропускаем через финальный линейный слой
        output = self.W_o(context)
        return output

# Проверка:
mha = CustomMultiHeadAttention(embed_dim=512, num_heads=8)
x = torch.randn(2, 10, 512) # 2 предложения, 10 слов, эмбеддинг 512
out = mha(x)

print("Смысл текста после Multi-Head Attention:", out.shape) # [2, 10, 512]

Смысл текста после Multi-Head Attention: torch.Size([2, 10, 512])


In [4]:
# Инициализация слоя
mha_layer = nn.MultiheadAttention(embed_dim=512, num_heads=8, batch_first=True)

# Прямой проход (обрати внимание, мы передаем x три раза: как Query, как Key и как Value)
attn_output, attn_weights = mha_layer(query=x, key=x, value=x)
print(attn_output.shape)

torch.Size([2, 10, 512])
